#### Import numpy & keras

In [7]:
import numpy as np
import keras
from datasets import load_dataset, DatasetDict, Image
import cv2
import datetime

In [8]:
import tensorflow as tf
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

2.21.0
Num GPUs Available:  0
[]


### Bilder normalisieren

In [9]:
def transform(example):
    image = np.array(example["image"], dtype=np.float32) / 255.0
    return {"image": image, "label": example["label"]}


#### 1. get training data

In [10]:
import matplotlib.pyplot as plt
import PIL
print(PIL.__version__)

ds = load_dataset("jonathan-roberts1/NWPU-RESISC45")
print(ds.shape)

train_data = ds["train"]
split_1 = train_data.train_test_split(
    test_size=0.15,
    seed=42,          # sorgt dafür, dass Validation immer gleich bleibt
    shuffle=True
)

validation_dataset = split_1["test"]
remaining_dataset = split_1["train"]


split_2 = remaining_dataset.train_test_split(
    test_size=0.25,
    seed=42,
    shuffle=True
)

train_ds = split_2["train"]
test_ds = split_2["test"]

# DatasetDict erzeugen
final_dataset = DatasetDict({
    "train": train_ds,
    "validation": validation_dataset,
    "test": test_ds
})
final_dataset = final_dataset.cast_column("image", Image())
final_dataset.with_format("tf")
final_dataset = final_dataset.with_transform(transform)

print(final_dataset)

train_images = (final_dataset["train"]["image"])
test_images = (final_dataset["test"]["image"])

train_labels = (final_dataset["train"]["label"])
test_labels = (final_dataset["test"]["label"])

tf_train = final_dataset["train"].to_tf_dataset(
    columns=["image"],
    label_cols=["label"],
    batch_size=128,
    shuffle=True
)

tf_test = final_dataset["test"].to_tf_dataset(
    columns=["image"],
    label_cols=["label"],
    batch_size=128
)

class_names = train_labels
print(class_names)

img_shape = np.array(final_dataset["train"][0]["image"]).shape
print(img_shape)



12.2.0
{'train': (31500, 2)}
DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 20081
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 4725
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 6694
    })
})
Column([13, 36, 33, 7, 38, ...])
(256, 256, 3)


#### 2. define architecture

In [16]:
from keras import Sequential

def generateModel(inputShape):
    model = Sequential()
    # 1st conv layer
    model.add(keras.layers.Conv2D(32, kernel_size=3, activation='relu', input_shape=inputShape))
    model.add(keras.layers.MaxPool2D(pool_size=(2,2), strides=2))
    # 2nd conv layer
    model.add(keras.layers.Conv2D(64, kernel_size=3, activation='relu'))
    model.add(keras.layers.MaxPool2D(pool_size=(2,2), strides=2))
    # fully connected layer
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(units=500, activation='relu'))
    model.add(keras.layers.Dense(units=45, activation='softmax'))
    return model

model = generateModel(img_shape)
#model = keras.models.load_model("./models/example_from_presentation.keras")
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 246016)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 500)            │   123,008,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 45)             │        22,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 123,050,439 (469.40 MB)

 Trainable params: 123,050,437 (469.40 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

#### 3. set training parameter and fit model

In [13]:
model.compile(
    optimizer='sgd',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=1,
    callbacks=[tensorboard_callback]
)
model.save("./models/example_from_presentation.keras")

157/157 ━━━━━━━━━━━━━━━━━━━━ 201s 1s/step - accuracy: 0.0437 - loss: 3.7303 - val_accuracy: 0.0811 - val_loss: 3.5785


In [15]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 41220), started 1:25:20 ago. (Use '!kill 41220' to kill it.)

#### 4. predict output 

In [14]:
prediction = model.predict(test_images[:1])

predicted_idx = np.argmax(prediction)

print(predicted_idx)
print(class_names[predicted_idx])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
12
4
